# Import e caricamento dei dati

In [1]:
import pickle
import os
import sys
from collections import defaultdict
from tqdm import tqdm

sys.path.insert(0, os.path.abspath('..'))
from Sorgente.Preprocessing.preprocesing import Preprocessore

In [2]:
with open('corpus_fever_arricchito.pkl', 'rb') as f:
    corpus_arricchito = pickle.load(f)
print(f'Documenti nel corpus: {len(corpus_arricchito):,}')

Documenti nel corpus: 14,364


In [3]:
preprocessore = Preprocessore()

In [4]:
DIMENSIONE_CAMPIONE = 1000
titoli_campione = list(corpus_arricchito.keys())[:DIMENSIONE_CAMPIONE]
corpus_campione = {t: corpus_arricchito[t] for t in titoli_campione}
print(f'Campione di test: {len(corpus_campione):,} documenti')

Campione di test: 1,000 documenti


In [5]:
def estrai_frase(lines: str, sentence_id: int) -> str:
    if not lines:
        return ''
    for riga in lines.split('\n'):
        riga = riga.strip()
        if not riga:
            continue
        parti = riga.split('\t', 1)
        if len(parti) == 2:
            try:
                if int(parti[0]) == sentence_id:
                    return parti[1].strip()
            except ValueError:
                continue
    return ''

# Struttura della Posting List

Prima di costruire l'indice si definisce la struttura dati che rappresenta una singola voce dell'indice

Si considera come Posting List un dizionario definito come *id_documento -> frequenza*


In [6]:
documenti = [
    (titolo, doc['text'])
    for titolo, doc in corpus_campione.items()
    if doc['text']
]
print(f'Documenti con testo: {len(documenti):,}')

Documenti con testo: 999


In [7]:
indice_prova = defaultdict(lambda: defaultdict(int))

for titolo, testo in documenti[:3]:
    token = preprocessore.elabora(testo)
    for termine in token:
        indice_prova[termine][titolo] += 1

print('Struttura indice primi 10 esempi:')
for termine, posting in list(indice_prova.items())[:10]:
    print(f'  "{termine}"')
    for id_doc, freq in posting.items():
        print(f'    → {id_doc:<40} freq={freq}')

Struttura indice primi 10 esempi:
  "19"
    → 19_Kids_and_Counting                     freq=2
  "kid"
    → 19_Kids_and_Counting                     freq=3
  "count"
    → 19_Kids_and_Counting                     freq=4
  "formerli"
    → 19_Kids_and_Counting                     freq=1
  "17"
    → 19_Kids_and_Counting                     freq=1
  "18"
    → 19_Kids_and_Counting                     freq=1
  "american"
    → 19_Kids_and_Counting                     freq=1
  "realiti"
    → 19_Kids_and_Counting                     freq=2
  "televis"
    → 19_Kids_and_Counting                     freq=2
    → 16th_IIFA_Awards                         freq=1
  "show"
    → 19_Kids_and_Counting                     freq=9
    → 16th_IIFA_Awards                         freq=1


Verifica su termine specifico

In [8]:
termine_cercato = 'american'
if termine_cercato in indice_prova:
    print(f'Posting list per "{termine_cercato}":')
    for id_doc, freq in indice_prova[termine_cercato].items():
        print(f'  {id_doc:<45} freq={freq}')
else:
    print(f'Termine "{termine_cercato}" non trovato nei 3 documenti di prova.')
    print('Prova con un termine presente nei tuoi documenti.')

Posting list per "american":
  19_Kids_and_Counting                          freq=1


# SPIMI (Single Pass In Memory Indexing)

SPIMI costruisce l'indice in un singolo passaggio sul corpus dividendolo in blocchi per non saturare la RAM

In [9]:
def costruisci_blocco_spimi(documenti_blocco: list[tuple[str, str]], preprocessore: Preprocessore) -> dict:
    blocco = defaultdict(lambda: defaultdict(int))
    for id_doc, testo in documenti_blocco:
        token = preprocessore.elabora(testo)
        for termine in token:
            blocco[termine][id_doc] += 1
    return dict(sorted(blocco.items()))

In [10]:
def salva_blocco(blocco: dict, percorso: str) -> None:
    with open(percorso, 'wb') as f:
        pickle.dump(blocco, f)

In [11]:
def carica_blocco(percorso: str) -> dict:
    with open(percorso, 'rb') as f:
        return pickle.load(f)

Test SPIMI su 1000 documenti in blocchi da 200

In [12]:
DIMENSIONE_BLOCCO = 200
CARTELLA_BLOCCHI = '../dati/elaborati/blocchi_spimi/'
os.makedirs(CARTELLA_BLOCCHI, exist_ok=True)

percorsi_blocchi = []

print(f'Costruzione indice SPIMI su {len(documenti):,} documenti')
print(f'Dimensione blocco: {DIMENSIONE_BLOCCO} documenti')
print(f'Blocchi attesi: {len(documenti) // DIMENSIONE_BLOCCO + 1}')
print()

for i in range(0, len(documenti), DIMENSIONE_BLOCCO):
    sotto_blocco = documenti[i: i + DIMENSIONE_BLOCCO]
    numero_blocco = i // DIMENSIONE_BLOCCO
    blocco = costruisci_blocco_spimi(sotto_blocco, preprocessore)
    percorso = os.path.join(CARTELLA_BLOCCHI, f'blocco_{numero_blocco:04d}.pkl')
    salva_blocco(blocco, percorso)
    percorsi_blocchi.append(percorso)
    print(f'  Blocco {numero_blocco:02d}: {len(sotto_blocco)} doc → {len(blocco):,} termini unici → salvato')

print(f'\nBlocchi creati: {len(percorsi_blocchi)}')

Costruzione indice SPIMI su 999 documenti
Dimensione blocco: 200 documenti
Blocchi attesi: 5

  Blocco 00: 200 doc → 4,776 termini unici → salvato
  Blocco 01: 200 doc → 5,370 termini unici → salvato
  Blocco 02: 200 doc → 5,691 termini unici → salvato
  Blocco 03: 200 doc → 6,400 termini unici → salvato
  Blocco 04: 199 doc → 6,319 termini unici → salvato

Blocchi creati: 5


# Merge dei blocchi e indice finale

Con SPIMI si hanno N blocchi parziali ordinati su disco da fondere poi tutti insieme in un unico indice finale. Viene fatto tramite coda di priorità che avanza su tutti i blocchi.

In [13]:
def merge_blocchi(percorsi_blocchi: list[str]) -> dict:
    indice_finale = defaultdict(lambda: defaultdict(int))
    for percorso in tqdm(percorsi_blocchi, desc='Merge blocchi'):
        blocco = carica_blocco(percorso)
        for termine, posting in blocco.items():
            for id_doc, freq in posting.items():
                indice_finale[termine][id_doc] += freq
    return dict(sorted(indice_finale.items()))

Test di esempio

In [14]:
indice = merge_blocchi(percorsi_blocchi)

print(f'\nIndice finale costruito:')
print(f'  Termini unici:  {len(indice):,}')
print(f'  Documenti:      {len(documenti):,}')

lunghezze_posting = [len(posting) for posting in indice.values()]
print(f'  Posting list media:  {sum(lunghezze_posting)/len(lunghezze_posting):.1f} doc per termine')
print(f'  Posting list max:    {max(lunghezze_posting)} doc (termine più comune)')
print(f'  Posting list min:    {min(lunghezze_posting)} doc (termine più raro)')

Merge blocchi: 100%|██████████| 5/5 [00:00<00:00, 62.31it/s]


Indice finale costruito:
  Termini unici:  15,237
  Documenti:      999
  Posting list media:  5.6 doc per termine
  Posting list max:    409 doc (termine più comune)
  Posting list min:    1 doc (termine più raro)


In [15]:
termini_comuni = sorted(indice.items(), key=lambda x: len(x[1]), reverse=True)

print('Top 15 termini più comuni:')
for termine, posting in termini_comuni[:15]:
    print(f'  "{termine}"  → {len(posting):,} documenti')

print()
print('Top 10 termini più rari:')
termini_rari = sorted(indice.items(), key=lambda x: len(x[1]))
for termine, posting in termini_rari[:10]:
    print(f'  "{termine}"  → {len(posting):,} documento')

Top 15 termini più comuni:
  "first"  → 409 documenti
  "also"  → 393 documenti
  "american"  → 381 documenti
  "film"  → 347 documenti
  "one"  → 306 documenti
  "award"  → 288 documenti
  "includ"  → 281 documenti
  "state"  → 273 documenti
  "time"  → 270 documenti
  "unit"  → 264 documenti
  "best"  → 260 documenti
  "year"  → 255 documenti
  "two"  → 243 documenti
  "world"  → 228 documenti
  "new"  → 221 documenti

Top 10 termini più rari:
  "002"  → 1 documento
  "005"  → 1 documento
  "02134"  → 1 documento
  "027"  → 1 documento
  "028"  → 1 documento
  "0445"  → 1 documento
  "05"  → 1 documento
  "0667"  → 1 documento
  "08"  → 1 documento
  "084640"  → 1 documento


# Verifica correttezza indice

In [16]:
def cerca_termine(indice: dict, termine: str, preprocessore: Preprocessore) -> None:
    stem = preprocessore.elabora(termine)
    if not stem:
        print(f'Termine "{termine}" rimosso dal preprocessing (stopword o vuoto).')
        return

    stem = stem[0]
    print(f'Termine: "{termine}" → stem: "{stem}"')

    if stem not in indice:
        print(f'  Termine non presente nell\'indice.')
        return

    posting = indice[stem]
    print(f'  Trovato in {len(posting):,} documenti:')
    for id_doc, freq in sorted(posting.items(), key=lambda x: -x[1])[:8]:
        print(f'    {id_doc:<45} freq={freq}')

In [17]:
for termine in ['American', 'film', 'award', 'basketball', 'born']:
    print()
    cerca_termine(indice, termine, preprocessore)


Termine: "American" → stem: "american"
  Trovato in 381 documenti:
    American_-LRB-word-RRB-                       freq=17
    American_literature                           freq=16
    American_Dad!                                 freq=11
    American_Airlines                             freq=10
    American_Pie_-LRB-film-RRB-                   freq=10
    Americans                                     freq=7
    American_Idiot                                freq=7
    American_Revolutionary_War                    freq=7

Termine: "film" → stem: "film"
  Trovato in 347 documenti:
    Aamir_Khan                                    freq=18
    A_Nightmare_on_Elm_Street                     freq=15
    Aditya_Chopra                                 freq=15
    Akshay_Kumar                                  freq=14
    2001-COLON-_A_Space_Odyssey_-LRB-film-RRB-    freq=13
    57th_National_Film_Awards                     freq=13
    Ajay_Devgn_filmography                        freq=13
    A

Verifica incrociata

In [18]:
titolo_verifica = list(corpus_campione.keys())[5]
testo_verifica  = corpus_campione[titolo_verifica]['text']
token_verifica  = preprocessore.elabora(testo_verifica)

print(f'VERIFICA INCROCIATA — documento: {titolo_verifica}')
print(f'Token del documento (primi 10): {token_verifica[:10]}')
print()
print('Controllo presenza nell\'indice:')
errori = 0
for termine in token_verifica[:20]:
    in_indice     = termine in indice
    doc_presente  = in_indice and titolo_verifica in indice[termine]
    print(f' "{termine}"  → doc nel indice: {doc_presente}')
    if not doc_presente:
        errori += 1

print(f'\nErrori: {errori} / 20')
if errori == 0:
    print('Indice corretto: tutti i termini del documento sono nell\'indice.')

VERIFICA INCROCIATA — documento: 1968_NBA_Playoffs
Token del documento (primi 10): ['1968', 'nba', 'playoff', 'postseason', 'tournament', 'nation', 'basketbal', 'associ', '196768', 'season']

Controllo presenza nell'indice:
 "1968"  → doc nel indice: True
 "nba"  → doc nel indice: True
 "playoff"  → doc nel indice: True
 "postseason"  → doc nel indice: True
 "tournament"  → doc nel indice: True
 "nation"  → doc nel indice: True
 "basketbal"  → doc nel indice: True
 "associ"  → doc nel indice: True
 "196768"  → doc nel indice: True
 "season"  → doc nel indice: True
 "tournament"  → doc nel indice: True
 "conclud"  → doc nel indice: True
 "eastern"  → doc nel indice: True
 "divis"  → doc nel indice: True
 "champion"  → doc nel indice: True
 "boston"  → doc nel indice: True
 "celtic"  → doc nel indice: True
 "defeat"  → doc nel indice: True
 "western"  → doc nel indice: True
 "divis"  → doc nel indice: True

Errori: 0 / 20
Indice corretto: tutti i termini del documento sono nell'indice.


# Logaritmic Merge

Il logaritmic merge gestisce l'aggiornamento dell'indice quando arrivano nuovi documenti senza ricostruire tutto.

In [19]:
class MergeLogaritmico:

    def __init__(self, preprocessore: Preprocessore):
        self.preprocessore = preprocessore
        self.livelli = []
        self.dimensioni = []
        self.documenti_totali = 0

    def _costruisci_indice_documento( self, id_doc: str, testo: str) -> dict:
        indice_doc = defaultdict(lambda: defaultdict(int))
        token = self.preprocessore.elabora(testo)
        for termine in token:
            indice_doc[termine][id_doc] += 1
        return dict(indice_doc)

    def _fondi_due_indici(self, indice_a: dict, indice_b: dict) -> dict:
        risultato = defaultdict(lambda: defaultdict(int))
        for termine, posting in indice_a.items():
            for id_doc, freq in posting.items():
                risultato[termine][id_doc] += freq
        for termine, posting in indice_b.items():
            for id_doc, freq in posting.items():
                risultato[termine][id_doc] += freq
        return dict(risultato)

    def inserisci(self, id_doc: str, testo: str) -> None:
        nuovo_indice = self._costruisci_indice_documento(id_doc, testo)
        nuova_dim = 1
        i = 0
        while i < len(self.livelli) and self.livelli[i] is not None:
            nuovo_indice = self._fondi_due_indici(self.livelli[i], nuovo_indice)
            nuova_dim += self.dimensioni[i]
            self.livelli[i] = None
            self.dimensioni[i] = 0
            i += 1

        if i == len(self.livelli):
            self.livelli.append(nuovo_indice)
            self.dimensioni.append(nuova_dim)
        else:
            self.livelli[i]    = nuovo_indice
            self.dimensioni[i] = nuova_dim

        self.documenti_totali += 1

    def ottieni_indice_completo(self) -> dict:
        indice_completo = {}
        for livello in self.livelli:
            if livello is not None:
                indice_completo = self._fondi_due_indici(indice_completo, livello)
        return indice_completo

    def stato(self) -> None:
        print(f'Documenti totali inseriti: {self.documenti_totali}')
        print(f'Livelli attivi:')
        for i, (livello, dim) in enumerate(zip(self.livelli, self.dimensioni)):
            stato = f'{dim} doc' if livello is not None else 'vuoto'
            print(f'  Z{i} (max {2**i:>4} doc): {stato}')

Test Logarithmic Merge su 20 documenti simulando lo stream

In [20]:
print('Test logarithmic merge su 20 documenti')
merge = MergeLogaritmico(preprocessore)

documenti_stream = documenti[800:820]

for i, (id_doc, testo) in enumerate(documenti_stream):
    merge.inserisci(id_doc, testo)
    if (i + 1) in [1, 2, 4, 8, 16, 20]:
        print(f'\nDopo {i+1} inserimenti:')
        merge.stato()

Test logarithmic merge su 20 documenti

Dopo 1 inserimenti:
Documenti totali inseriti: 1
Livelli attivi:
  Z0 (max    1 doc): 1 doc

Dopo 2 inserimenti:
Documenti totali inseriti: 2
Livelli attivi:
  Z0 (max    1 doc): vuoto
  Z1 (max    2 doc): 2 doc

Dopo 4 inserimenti:
Documenti totali inseriti: 4
Livelli attivi:
  Z0 (max    1 doc): vuoto
  Z1 (max    2 doc): vuoto
  Z2 (max    4 doc): 4 doc

Dopo 8 inserimenti:
Documenti totali inseriti: 8
Livelli attivi:
  Z0 (max    1 doc): vuoto
  Z1 (max    2 doc): vuoto
  Z2 (max    4 doc): vuoto
  Z3 (max    8 doc): 8 doc

Dopo 16 inserimenti:
Documenti totali inseriti: 16
Livelli attivi:
  Z0 (max    1 doc): vuoto
  Z1 (max    2 doc): vuoto
  Z2 (max    4 doc): vuoto
  Z3 (max    8 doc): vuoto
  Z4 (max   16 doc): 16 doc

Dopo 20 inserimenti:
Documenti totali inseriti: 20
Livelli attivi:
  Z0 (max    1 doc): vuoto
  Z1 (max    2 doc): vuoto
  Z2 (max    4 doc): 4 doc
  Z3 (max    8 doc): vuoto
  Z4 (max   16 doc): 16 doc


In [21]:
indice_merge = merge.ottieni_indice_completo()

print('Verifica correttezza Merge Logaritmico')
print(f'Documenti inseriti: {merge.documenti_totali}')
print(f'Termini unici nell\'indice: {len(indice_merge):,}')
print()

errori = 0
for id_doc, testo in documenti_stream[:5]:
    token = preprocessore.elabora(testo)
    if not token:
        continue
    termine_test = token[0]
    presente = termine_test in indice_merge and id_doc in indice_merge[termine_test]
    print(f'  {id_doc:<40} termine "{termine_test}" presente: {presente}')
    if not presente:
        errori += 1

print(f'\nErrori: {errori}')

Verifica correttezza Merge Logaritmico
Documenti inseriti: 20
Termini unici nell'indice: 1,096

  Albanians                                termine "albanian" presente: True
  Alan_White                               termine "alan" presente: True
  All_Eyez_on_Me                           termine "eyez" presente: True
  American_Gothic_-LRB-2016_TV_series-RRB- termine "american" presente: True
  Alpha_Flight                             termine "alpha" presente: True

Errori: 0


# Costruzione indice completo

In [22]:
PERCORSO_INDICE_FINALE = '../dati/elaborati/indice_fever.pkl'
os.makedirs('../dati/elaborati/', exist_ok=True)

if os.path.exists(PERCORSO_INDICE_FINALE):
    print('Indice già salvato. Carico da disco...')
    with open(PERCORSO_INDICE_FINALE, 'rb') as f:
        indice_completo = pickle.load(f)
    print(f'Caricato: {len(indice_completo):,} termini')
else:
    print('Costruzione indice completo su tutto il corpus...')
    tutti_i_documenti = [
        (titolo, doc['text'])
        for titolo, doc in corpus_arricchito.items()
        if doc['text']
    ]
    print(f'Documenti totali: {len(tutti_i_documenti):,}')

    DIM_BLOCCO_FINALE = 500
    percorsi_blocchi_finali = []

    for i in tqdm(range(0, len(tutti_i_documenti), DIM_BLOCCO_FINALE),desc='SPIMI'):
        blocco_doc = tutti_i_documenti[i: i + DIM_BLOCCO_FINALE]
        numero = i // DIM_BLOCCO_FINALE
        blocco = costruisci_blocco_spimi(blocco_doc, preprocessore)
        percorso = os.path.join(CARTELLA_BLOCCHI, f'finale_{numero:04d}.pkl')
        salva_blocco(blocco, percorso)
        percorsi_blocchi_finali.append(percorso)

    indice_completo = merge_blocchi(percorsi_blocchi_finali)
    numero_doc = len(tutti_i_documenti)
    lunghezze_doc = {
        titolo: len(preprocessore.elabora(testo))
        for titolo, testo in tutti_i_documenti
    }
    stato_da_salvare = {
        'indice': indice_completo,
        'lunghezze_documenti': lunghezze_doc,
        'numero_documenti': numero_doc
    }
    with open(PERCORSO_INDICE_FINALE, 'wb') as f:
        pickle.dump(stato_da_salvare, f)

    dim_mb = os.path.getsize(PERCORSO_INDICE_FINALE) / 1024**2
    print(f'\nIndice salvato: {PERCORSO_INDICE_FINALE}')
    print(f'Termini unici: {len(indice_completo):,}')
    print(f'Dimensione:    {dim_mb:.0f} MB')

Indice già salvato. Carico da disco...
Caricato: 3 termini


In [26]:
indice_reale = indice_completo['indice']
lunghezze = [len(p) for p in indice_reale.values()]

print('Statistiche indice completo')
print(f'  Termini unici:          {len(indice_reale):,}')
print(f'  Media doc per termine:  {sum(lunghezze)/len(lunghezze):.1f}')
print(f'  Max doc per termine:    {max(lunghezze):,}')
print(f'  Min doc per termine:    {min(lunghezze)}')
print(f'  Termini in 1 solo doc:  {sum(1 for l in lunghezze if l == 1):,}')

Statistiche indice completo
  Termini unici:          71,837
  Media doc per termine:  17.6
  Max doc per termine:    6,204
  Min doc per termine:    1
  Termini in 1 solo doc:  38,057
